# 01 — Data Understanding

Notebook này giúp kiểm tra dữ liệu thô của Store Sales – Time Series Forecasting: cấu trúc, chất lượng, phạm vi thời gian, quan hệ giữa các bảng và mức độ sẵn có của thông tin khi dự báo.

**Phạm vi:** chỉ quan sát và lập báo cáo. Chưa điền missing, loại outlier, ghép bảng modeling, tạo lag/rolling, chia validation hay huấn luyện mô hình. Giá trị `sales = 0` được giữ nguyên, không mặc định là lỗi.


## 1. Project Overview

Bài toán dự báo `sales` theo ngày cho từng cặp cửa hàng (`store_nbr`) và nhóm sản phẩm (`family`). Dữ liệu gồm 7 CSV trong `data/raw`. Notebook tự tìm thư mục gốc của dự án từ thư mục đang chạy hoặc các thư mục cha.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)


Pandas: 3.1.1
NumPy: 2.5.4


## 2. Data Path Configuration

Nếu notebook được mở từ thư mục dự án hoặc `notebooks/`, phần này sẽ tìm thấy `data/raw/train.csv`. Nếu chạy từ vị trí không liên quan đến repo, hãy mở notebook từ thư mục dự án trong VS Code/Jupyter.


In [2]:
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "data" / "raw" / "train.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy data/raw/train.csv. Hãy mở notebook từ thư mục dự án "
        "hoặc thư mục con của dự án."
    )

DATA_DIR = ROOT / "data" / "raw"
REPORTS_DIR = ROOT / "reports" / "understanding"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
print("Current directory:", Path.cwd())
print("Project root:", ROOT)
print("Data directory:", DATA_DIR)
print("Reports directory:", REPORTS_DIR)


Current directory: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-\notebooks
Project root: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-
Data directory: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-\data\raw
Reports directory: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-\reports\understanding


## 3. Load All 7 Datasets

Trước tiên kiểm tra đủ file, sau đó đọc dữ liệu. `date` được parse thành kiểu ngày để thống kê thời gian; đây không phải thao tác sửa dữ liệu.


In [3]:
REQUIRED_FILES = [
    "train.csv", "test.csv", "stores.csv", "oil.csv",
    "holidays_events.csv", "transactions.csv", "sample_submission.csv",
]
file_inventory = pd.DataFrame([
    {
        "filename": name,
        "exists": (DATA_DIR / name).is_file(),
        "size_MB": round((DATA_DIR / name).stat().st_size / (1024 ** 2), 2)
        if (DATA_DIR / name).is_file() else np.nan,
    }
    for name in REQUIRED_FILES
])
display(file_inventory)
missing_files = file_inventory.loc[~file_inventory["exists"], "filename"].tolist()
if missing_files:
    raise FileNotFoundError(f"Thi?u file trong {DATA_DIR}: {missing_files}")

read_options = {"low_memory": False}
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["date"], **read_options)
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["date"], **read_options)
stores = pd.read_csv(DATA_DIR / "stores.csv", **read_options)
oil = pd.read_csv(DATA_DIR / "oil.csv", parse_dates=["date"], **read_options)
holidays = pd.read_csv(DATA_DIR / "holidays_events.csv", parse_dates=["date"], **read_options)
transactions = pd.read_csv(DATA_DIR / "transactions.csv", parse_dates=["date"], **read_options)
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", **read_options)

datasets = {
    "train": train, "test": test, "stores": stores, "oil": oil,
    "holidays": holidays, "transactions": transactions,
    "sample_submission": sample_submission,
}
print(f"Loaded {len(datasets)} datasets")
for name, df in datasets.items():
    print(f"{name:20s} {df.shape}")


,filename,exists,size_MB
0,train.csv,True,116.16
1,test.csv,True,0.97
2,stores.csv,True,0.00
3,oil.csv,True,0.02
4,holidays_events.csv,True,0.02
5,transactions.csv,True,1.48
6,sample_submission.csv,True,0.33


Loaded 7 datasets
train                (3000888, 6)
test                 (28512, 5)
stores               (54, 5)
oil                  (1218, 2)
holidays             (350, 6)
transactions         (83488, 3)
sample_submission    (28512, 2)


## 4. Dataset Overview: Shape, Columns, Types, Memory

Các bảng dưới đây mô tả dữ liệu được đọc nguyên trạng. `n_unique` không bao gồm giá trị thiếu.


In [4]:
shape_report = pd.DataFrame([
    {
        "dataset": name, "rows": len(df), "columns": df.shape[1],
        "memory_MB": round(df.memory_usage(index=True, deep=True).sum() / (1024 ** 2), 2),
    }
    for name, df in datasets.items()
])
display(shape_report)

for name, df in datasets.items():
    print(f"\n{'=' * 72}\n{name.upper()}\n{'=' * 72}")
    print("Columns:", df.columns.tolist())
    display(df.head(3))
    df.info(memory_usage="deep")

dtype_report = pd.DataFrame([
    {"dataset": name, "column": col, "dtype": str(df[col].dtype),
     "n_unique": int(df[col].nunique(dropna=True))}
    for name, df in datasets.items() for col in df.columns
])
display(dtype_report)


,dataset,rows,columns,memory_MB
0,train,3000888,6,168.16
1,test,28512,5,1.38
2,stores,54,5,0.00
3,oil,1218,2,0.02
4,holidays,350,6,0.03
5,transactions,83488,3,1.91
6,sample_submission,28512,2,0.44



TRAIN
Columns: ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion']


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0


<class 'pandas.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype         
---  ------       -----         
 0   id           int64         
 1   date         datetime64[us]
 2   store_nbr    int64         
 3   family       str           
 4   sales        float64       
 5   onpromotion  int64         
dtypes: datetime64[us](1), float64(1), int64(3), str(1)
memory usage: 168.2 MB

TEST
Columns: ['id', 'date', 'store_nbr', 'family', 'onpromotion']


,id,date,store_nbr,family,onpromotion
0,3000888,2017-08-16,1,AUTOMOTIVE,0
1,3000889,2017-08-16,1,BABY CARE,0
2,3000890,2017-08-16,1,BEAUTY,2


<class 'pandas.DataFrame'>
RangeIndex: 28512 entries, 0 to 28511
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id           28512 non-null  int64         
 1   date         28512 non-null  datetime64[us]
 2   store_nbr    28512 non-null  int64         
 3   family       28512 non-null  str           
 4   onpromotion  28512 non-null  int64         
dtypes: datetime64[us](1), int64(3), str(1)
memory usage: 1.4 MB

STORES
Columns: ['store_nbr', 'city', 'state', 'type', 'cluster']


... (7 outputs left out)

<class 'pandas.DataFrame'>
RangeIndex: 83488 entries, 0 to 83487
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          83488 non-null  datetime64[us]
 1   store_nbr     83488 non-null  int64         
 2   transactions  83488 non-null  int64         
dtypes: datetime64[us](1), int64(2)
memory usage: 1.9 MB

SAMPLE_SUBMISSION
Columns: ['id', 'sales']


,id,sales
0,3000888,0.0
1,3000889,0.0
2,3000890,0.0


<class 'pandas.DataFrame'>
RangeIndex: 28512 entries, 0 to 28511
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      28512 non-null  int64  
 1   sales   28512 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 445.6 KB


,dataset,column,dtype,n_unique
0,train,id,int64,3000888
1,train,date,datetime64[us],1684
2,train,store_nbr,int64,54
3,train,family,str,33
4,train,sales,float64,379610
5,train,onpromotion,int64,362
6,test,id,int64,28512
7,test,date,datetime64[us],16
8,test,store_nbr,int64,54
9,test,family,str,33


## 5. Time Coverage and Date Continuity

Bảng thời gian chỉ áp dụng cho các dataset có cột `date`. Phần kiểm tra tính liên tục ghi nhận ngày bị thiếu; notebook không tự tạo hoặc điền ngày.


In [5]:
date_rows = []
for name, df in datasets.items():
    if "date" in df.columns:
        dates = df["date"].dropna()
        date_rows.append({
            "dataset": name, "start_date": dates.min(), "end_date": dates.max(),
            "n_unique_dates": dates.nunique(),
        })
date_report = pd.DataFrame(date_rows)
display(date_report)

continuity_rows = []
for name, df in datasets.items():
    if "date" in df.columns and df["date"].notna().any():
        present = pd.DatetimeIndex(df["date"].dropna().unique()).sort_values()
        expected = pd.date_range(present.min(), present.max(), freq="D")
        absent = expected.difference(present)
        continuity_rows.append({
            "dataset": name, "calendar_days_in_range": len(expected),
            "dates_present": len(present), "dates_absent": len(absent),
            "first_absent_dates": ", ".join(str(x.date()) for x in absent[:10]),
        })
date_continuity_report = pd.DataFrame(continuity_rows)
display(date_continuity_report)

records_per_day = (
    pd.concat([
        train.groupby("date").size().rename("train_rows"),
        test.groupby("date").size().rename("test_rows"),
    ], axis=1).fillna(0).astype(int).rename_axis("date").reset_index()
)
print("Train rows per date:")
display(records_per_day.loc[records_per_day.train_rows > 0, "train_rows"].describe())
print("Test rows per date:")
display(records_per_day.loc[records_per_day.test_rows > 0, "test_rows"].describe())


,dataset,start_date,end_date,n_unique_dates
0,train,2013-01-01,2017-08-15,1684
1,test,2017-08-16,2017-08-31,16
2,oil,2013-01-01,2017-08-31,1218
3,holidays,2012-03-02,2017-12-26,312
4,transactions,2013-01-01,2017-08-15,1682


,dataset,calendar_days_in_range,dates_present,dates_absent,first_absent_dates
0,train,1688,1684,4,"2013-12-25, 2014-12-25, 2015-12-25, 2016-12-25"
1,test,16,16,0,
2,oil,1704,1218,486,"2013-01-05, 2013-01-06, 2013-01-12, 2013-01-13..."
3,holidays,2126,312,1814,"2012-03-03, 2012-03-04, 2012-03-05, 2012-03-06..."
4,transactions,1688,1682,6,"2013-12-25, 2014-12-25, 2015-12-25, 2016-01-01..."


Train rows per date:


count    1684.0
mean     1782.0
std         0.0
min      1782.0
25%      1782.0
50%      1782.0
75%      1782.0
max      1782.0
Name: train_rows, dtype: float64

Test rows per date:


count      16.0
mean     1782.0
std         0.0
min      1782.0
25%      1782.0
50%      1782.0
75%      1782.0
max      1782.0
Name: test_rows, dtype: float64

## 6. Missing Values

Missing được thống kê riêng theo file và cột. Notebook không điền giá dầu hoặc giá trị nào khác.


In [6]:
missing_report = pd.DataFrame([
    {"dataset": name, "column": col, "missing_count": int(df[col].isna().sum()),
     "missing_pct": round(float(df[col].isna().mean() * 100), 4)}
    for name, df in datasets.items() for col in df.columns
])
missing_report = missing_report.sort_values(["dataset", "missing_count"], ascending=[True, False])
display(missing_report.loc[missing_report.missing_count > 0])
if not (missing_report.missing_count > 0).any():
    print("Kh?ng ph?t hi?n missing ? c?c c?t.")


,dataset,column,missing_count,missing_pct
17,oil,dcoilwtico,43,3.5304


## 7. Duplicate Analysis

Kiểm tra cả dòng trùng hoàn toàn và khóa nghiệp vụ. Nhiều sự kiện ngày lễ cùng ngày không mặc định là duplicate; notebook không áp đặt khóa duy nhất cho bảng holiday.


In [7]:
business_keys = {
    "train": ["date", "store_nbr", "family"],
    "test": ["date", "store_nbr", "family"],
    "stores": ["store_nbr"],
    "oil": ["date"],
    "transactions": ["date", "store_nbr"],
    "sample_submission": ["id"],
}
duplicate_rows = []
for name, df in datasets.items():
    duplicate_rows.append({
        "dataset": name, "duplicate_type": "exact_rows",
        "key_columns": "all columns", "duplicate_count": int(df.duplicated().sum()),
    })
    if name in business_keys:
        keys = business_keys[name]
        duplicate_rows.append({
            "dataset": name, "duplicate_type": "business_key",
            "key_columns": ", ".join(keys),
            "duplicate_count": int(df.duplicated(subset=keys).sum()),
        })
duplicate_report = pd.DataFrame(duplicate_rows)
display(duplicate_report)


,dataset,duplicate_type,key_columns,duplicate_count
0,train,exact_rows,all columns,0
1,train,business_key,"date, store_nbr, family",0
2,test,exact_rows,all columns,0
3,test,business_key,"date, store_nbr, family",0
4,stores,exact_rows,all columns,0
5,stores,business_key,store_nbr,0
6,oil,exact_rows,all columns,0
7,oil,business_key,date,0
8,holidays,exact_rows,all columns,0
9,transactions,exact_rows,all columns,0


## 8. Dataset Relationships and Key Coverage

Đây là kiểm tra khóa và độ phủ, không thực hiện merge. Ngày lễ có thể có nhiều sự kiện cùng ngày và cần được xử lý theo phạm vi National/Regional/Local trong notebook preprocessing.


In [8]:
train_stores = set(train.store_nbr.unique())
test_stores = set(test.store_nbr.unique())
meta_stores = set(stores.store_nbr.unique())
train_families = set(train.family.unique())
test_families = set(test.family.unique())

relationship_report = pd.DataFrame([
    {"check": "train store IDs absent from stores.csv", "count": len(train_stores - meta_stores)},
    {"check": "test store IDs absent from stores.csv", "count": len(test_stores - meta_stores)},
    {"check": "train families absent from test", "count": len(train_families - test_families)},
    {"check": "test families absent from train", "count": len(test_families - train_families)},
    {"check": "transaction store IDs absent from stores.csv", "count": len(set(transactions.store_nbr.unique()) - meta_stores)},
    {"check": "oil dates overlapping train", "count": len(set(oil.date.dropna()) & set(train.date.unique()))},
    {"check": "oil dates overlapping test", "count": len(set(oil.date.dropna()) & set(test.date.unique()))},
    {"check": "holiday event records on train dates", "count": int(holidays.date.isin(train.date).sum())},
    {"check": "holiday event records on test dates", "count": int(holidays.date.isin(test.date).sum())},
])
display(relationship_report)

print("Store IDs in train:", sorted(train_stores))
print("Family values in train:", sorted(train_families))
print("Holiday locale counts:")
display(holidays["locale"].value_counts(dropna=False).rename_axis("locale").to_frame("records"))
print("Holiday event types:")
display(holidays["type"].value_counts(dropna=False).rename_axis("type").to_frame("records"))

submission_alignment = {
    "same_row_count_as_test": len(sample_submission) == len(test),
    "same_id_set_as_test": set(sample_submission.id) == set(test.id),
    "same_id_order_as_test": sample_submission.id.equals(test.id),
}
display(pd.Series(submission_alignment, name="passed").to_frame())


,check,count
0,train store IDs absent from stores.csv,0
1,test store IDs absent from stores.csv,0
2,train families absent from test,0
3,test families absent from train,0
4,transaction store IDs absent from stores.csv,0
5,oil dates overlapping train,1203
6,oil dates overlapping test,12
7,holiday event records on train dates,282
8,holiday event records on test dates,1


Store IDs in train: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54)]
Family values in train: ['AUTOMOTIVE', 'BABY CARE', 'BEAUTY', 'BEVERAGES', 'BOOKS', 'BREAD/BAKERY', 'CELEBRATION', 'CLEANING', 'DAIRY', 'DELI', 'EGGS', 'FROZEN FOODS', 'GROCERY I', 'GROCERY II', 'HARDWARE', 'HOME AND KITCHEN I', 'HO

,records
locale,
National,174
Local,152
Regional,24


Holiday event types:


,records
type,
Holiday,221
Event,56
Additional,51
Transfer,12
Bridge,5
Work Day,5


,passed
same_row_count_as_test,True
same_id_set_as_test,True
same_id_order_as_test,True


## 9. Target Sanity Check

Thống kê sơ bộ cho `sales`. Giá trị 0 và số thập phân được mô tả, không tự động coi là lỗi.


In [9]:
sales_summary = train["sales"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
).to_frame("sales")
display(sales_summary)

sales_quality = pd.DataFrame([
    {"metric": "total_rows", "value": len(train), "percentage": 100.0},
    {"metric": "zero_sales", "value": int(train.sales.eq(0).sum()), "percentage": train.sales.eq(0).mean() * 100},
    {"metric": "negative_sales", "value": int(train.sales.lt(0).sum()), "percentage": train.sales.lt(0).mean() * 100},
    {"metric": "positive_sales", "value": int(train.sales.gt(0).sum()), "percentage": train.sales.gt(0).mean() * 100},
    {"metric": "missing_sales", "value": int(train.sales.isna().sum()), "percentage": train.sales.isna().mean() * 100},
])
display(sales_quality.round(4))

family_summary = (train.groupby("family").sales.agg(
    total_sales="sum", mean_sales="mean", median_sales="median", std_sales="std",
    zero_ratio=lambda x: x.eq(0).mean()
).sort_values("total_sales", ascending=False))
store_summary = (train.groupby("store_nbr").sales.agg(
    total_sales="sum", mean_sales="mean", median_sales="median", std_sales="std",
    zero_ratio=lambda x: x.eq(0).mean()
).sort_values("total_sales", ascending=False))
print("Sales summary by family:")
display(family_summary)
print("Sales summary by store:")
display(store_summary)


,sales
count,3.000888e+06
mean,3.577757e+02
std,1.101998e+03
min,0.000000e+00
1%,0.000000e+00
5%,0.000000e+00
25%,0.000000e+00
50%,1.100000e+01
75%,1.958473e+02
90%,8.670000e+02


,metric,value,percentage
0,total_rows,3000888,100.0000
1,zero_sales,939130,31.2951
2,negative_sales,0,0.0000
3,positive_sales,2061758,68.7049
4,missing_sales,0,0.0000


Sales summary by family:


,total_sales,mean_sales,median_sales,std_sales,zero_ratio
family,,,,,
GROCERY I,3.434627e+08,3776.972100,3185.0000,2874.208845,0.080628
BEVERAGES,2.169545e+08,2385.793151,1784.0000,2307.882305,0.080617
PRODUCE,1.227047e+08,1349.352123,398.2905,2186.481332,0.283551
CLEANING,9.752129e+07,1072.416744,938.0000,734.681493,0.080617
DAIRY,6.448771e+07,709.154889,520.0000,671.949638,0.080617
BREAD/BAKERY,4.213395e+07,463.336254,401.0000,368.246367,0.080606
POULTRY,3.187600e+07,350.532292,205.7430,400.511631,0.080760
MEATS,3.108647e+07,341.849965,224.9365,455.908498,0.080617
PERSONAL CARE,2.459205e+07,270.432513,222.0000,226.512007,0.080650


Sales summary by store:


,total_sales,mean_sales,median_sales,std_sales,zero_ratio
store_nbr,,,,,
44,6.208755e+07,1117.245254,75.0000,2685.282436,0.179479
45,5.449801e+07,980.673908,62.0000,2384.321416,0.193587
47,5.094831e+07,916.798209,51.0000,2222.316800,0.189538
3,5.048191e+07,908.405495,64.0000,2146.185033,0.181404
49,4.342010e+07,781.330450,48.0000,1897.861943,0.193479
46,4.189606e+07,753.905962,36.0000,1889.012915,0.193209
48,3.593313e+07,646.604950,30.0000,1668.493989,0.213219
51,3.291149e+07,592.231511,27.0000,1352.209195,0.201216
8,3.049429e+07,548.734739,39.0000,1275.423965,0.199309


## 10. Data Availability for Forecasting

| Dataset/field | Có trong giai đoạn test? | Ý nghĩa và lưu ý |
|---|---|---|
| `test.date`, `store_nbr`, `family` | Có | Các khóa và ngày cần dự báo. |
| `test.onpromotion` | Có | Promotion cho test được cung cấp. |
| `train.sales` | Không | Dữ liệu mục tiêu lịch sử; khi validation phải tránh dùng doanh số thật của ngày đang dự báo. |
| `transactions.csv` | Không có cho test | Chỉ có đến cuối train; không dùng giao dịch thực tế tương lai. |
| `stores.csv` | Có thể dùng | Metadata cửa hàng tĩnh. |
| `holidays_events.csv` | Có sự kiện quanh giai đoạn test | Cần xét ngày và phạm vi địa lý, tránh nhân bản dòng. |
| `oil.csv` | File có ngày đến 2017-08-31 | Cần xác định giả định khả dụng và áp dụng nhất quán cho validation. |

Bảng này ghi nhận dữ liệu trong các file được cung cấp. Một giá trị có trong file không tự động chứng minh rằng nó đã biết tại thời điểm dự báo thực tế; quy tắc này cần được chốt khi thiết kế validation.


## 11. Data Integrity Summary

Các kiểm tra dưới đây báo cáo trạng thái, không tự xóa hoặc sửa dòng. Nếu `passed=False`, hãy điều tra nguyên nhân trước khi xử lý.


In [10]:
n_stores = train.store_nbr.nunique()
n_families = train.family.nunique()
n_series = train[["store_nbr", "family"]].drop_duplicates().shape[0]
expected_series = n_stores * n_families

# Compare every series with the dates actually observed in train.
expected_train_dates = train.date.nunique()
series_date_check = (
    train.groupby(["store_nbr", "family"])["date"]
    .agg(first_date="min", last_date="max", n_dates="nunique")
    .reset_index()
)
incomplete_series = series_date_check[
    (series_date_check["first_date"] != train.date.min())
    | (series_date_check["last_date"] != train.date.max())
    | (series_date_check["n_dates"] != expected_train_dates)
]

series_per_date = train.groupby("date").apply(
    lambda group: group[["store_nbr", "family"]].drop_duplicates().shape[0]
)

checks = {
    "all_7_required_files_exist": len(datasets) == 7 and file_inventory.exists.all(),
    "train_date_is_datetime": pd.api.types.is_datetime64_any_dtype(train.date),
    "test_date_is_datetime": pd.api.types.is_datetime64_any_dtype(test.date),
    "train_business_key_unique": not train.duplicated(["date", "store_nbr", "family"]).any(),
    "test_business_key_unique": not test.duplicated(["date", "store_nbr", "family"]).any(),
    "store_key_unique": stores.store_nbr.is_unique,
    "oil_date_unique": oil.date.is_unique,
    "transaction_business_key_unique": not transactions.duplicated(["date", "store_nbr"]).any(),
    "submission_id_unique": sample_submission.id.is_unique,
    "submission_ids_match_test": set(sample_submission.id) == set(test.id),
    "train_sales_non_negative": bool(train.sales.dropna().ge(0).all()),
    "expected_store_family_count": n_series == expected_series,
    "all_train_dates_have_all_series": bool(series_per_date.eq(n_series).all()),
    "all_series_cover_observed_train_dates": incomplete_series.empty,
}
integrity_report = pd.DataFrame([
    {"check": name, "passed": bool(value)} for name, value in checks.items()
])
display(integrity_report)
print(f"Stores={n_stores}; families={n_families}; series={n_series}; product={n_stores * n_families}")
print("Observed train dates per complete series:", expected_train_dates)
print("Incomplete Store-Family series:", len(incomplete_series))
if not incomplete_series.empty:
    display(incomplete_series.head(20))
print("Dates with unexpected number of series:", int(series_per_date.ne(n_series).sum()))


,check,passed
0,all_7_required_files_exist,True
1,train_date_is_datetime,True
2,test_date_is_datetime,True
3,train_business_key_unique,True
4,test_business_key_unique,True
5,store_key_unique,True
6,oil_date_unique,True
7,transaction_business_key_unique,True
8,submission_id_unique,True
9,submission_ids_match_test,True


Stores=54; families=33; series=1782; product=1782
Observed train dates per complete series: 1684
Incomplete Store-Family series: 0
Dates with unexpected number of series: 0


## 12. Save Data Understanding Reports

Notebook lưu các bảng tổng hợp vào `reports/understanding/`. Đây là báo cáo dẫn xuất; dữ liệu CSV gốc không bị thay đổi.


In [11]:
reports_to_save = {
    "data_shape_report.csv": shape_report,
    "data_dtype_report.csv": dtype_report,
    "data_date_range_report.csv": date_report,
    "data_missing_report.csv": missing_report,
    "data_duplicate_report.csv": duplicate_report,
    "data_integrity_report.csv": integrity_report,
    "data_date_continuity_report.csv": date_continuity_report,
    "data_relationship_report.csv": relationship_report,
}
for filename, report in reports_to_save.items():
    report.to_csv(REPORTS_DIR / filename, index=False)
    print("Saved:", REPORTS_DIR / filename)


Saved: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-\reports\understanding\data_shape_report.csv
Saved: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-\reports\understanding\data_dtype_report.csv
Saved: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-\reports\understanding\data_date_range_report.csv
Saved: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-\reports\understanding\data_missing_report.csv
Saved: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-\reports\understanding\data_duplicate_report.csv
Saved: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-\reports\understanding\data_integrity_report.csv
Saved: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-\reports\understanding\data_date_continuity_report.csv
Saved: d:\STORE-SALES-TIME-SERIES-FORECASTING-KAGGLE-\reports\understanding\data_relationship_report.csv


## 13. Key Findings and Conclusion

Phần dưới đây được tạo từ dữ liệu thực tế khi chạy notebook. Hãy xem kết quả trước khi bắt đầu EDA và preprocessing.


In [ ]:
print("### Dataset Overview")
print(f"- Training records: {len(train):,}")
print(f"- Testing records: {len(test):,}")
print(f"- Stores / families / Store-Family series: {n_stores} / {n_families} / {n_series}")
print(f"- Training period: {train.date.min().date()} to {train.date.max().date()} ({train.date.nunique():,} days)")
print(f"- Testing period: {test.date.min().date()} to {test.date.max().date()} ({test.date.nunique():,} days)")
print("\n### Data Quality")
print(f"- Missing cells across all datasets: {int(missing_report.missing_count.sum()):,}")
print(f"- Exact duplicate rows across all datasets: {sum(df.duplicated().sum() for df in datasets.values()):,}")
print(f"- Incomplete Store-Family series: {len(incomplete_series):,}")
print(f"- Business-key duplicate records in train/test: {int(train.duplicated(['date','store_nbr','family']).sum()):,} / {int(test.duplicated(['date','store_nbr','family']).sum()):,}")
print("\n### Target")
print(f"- Sales min/median/mean/max: {train.sales.min():,.2f} / {train.sales.median():,.2f} / {train.sales.mean():,.2f} / {train.sales.max():,.2f}")
print(f"- Zero sales: {train.sales.eq(0).mean():.2%}; negative sales: {int(train.sales.lt(0).sum()):,}")
print("\n### Forecast Availability and Risks")
print("- Promotion and calendar/store/family keys are present in test or lookup files.")
print("- Transactions stop at the end of train; actual future transactions are unavailable.")
print("- Oil file includes dates through the end of test, but validation must follow a consistent availability assumption.")
print("- Holiday events can be multiple per date and vary by geographic locale.")
print("- Next step: visualize sales/time patterns, then decide cleaning and feature rules without treating zero as missing by default.")


### Dataset Overview
- Training records: 3,000,888
- Testing records: 28,512
- Stores / families / Store-Family series: 54 / 33 / 1782
- Training period: 2013-01-01 to 2017-08-15 (1,684 days)
- Testing period: 2017-08-16 to 2017-08-31 (16 days)

### Data Quality
- Missing cells across all datasets: 43
- Exact duplicate rows across all datasets: 0
- Incomplete Store-Family series: 0
- Business-key duplicate records in train/test: 0 / 0

### Target
- Sales min/median/mean/max: 0.00 / 11.00 / 357.78 / 124,717.00
- Zero sales: 31.30%; negative sales: 0

### Forecast Availability and Risks
- Promotion and calendar/store/family keys are present in test or lookup files.
- Transactions stop at the end of train; actual future transactions are unavailable.
- Oil file includes dates through the end of test, but validation must follow a consistent availability assumption.
- Holiday events can be multiple per date and vary by geographic locale.
- Next step: visualize sales/time patterns, then decid

: 